# 유동인구데이터 다운 자동화

In [7]:
!pip install -q -U selenium

In [ ]:
!apt-get update -qq
!apt-get install -y chromium chromium-driver

In [ ]:
import shutil

print("chromium:", shutil.which("chromium"))
print("chromedriver:", shutil.which("chromedriver"))

In [ ]:
import os

candidates = [
    "/usr/bin/chromium",
    "/usr/bin/chromium-browser",
    "/usr/bin/google-chrome",
    "/usr/bin/google-chrome-stable"
]

for path in candidates:
    print(path, "->", os.path.exists(path))

In [ ]:
!chromium-browser --version
!chromedriver --version

In [ ]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb

In [ ]:
!google-chrome --version

In [2]:
import os
import time
import zipfile
from datetime import datetime, timedelta

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

In [17]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()

options.binary_location = "/usr/bin/google-chrome"

options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(options=options)

브라우저 실행 성공


In [18]:
import time

URL = "https://data.seoul.go.kr/dataList/OA-22300/F/1/datasetView.do"

driver.get(URL)

time.sleep(5)

print("페이지 제목:", driver.title)
print("현재 URL:", driver.current_url)

페이지 제목: 수도권 생활이동 (출발-도착지 기준)> 데이터셋> 공공데이터 | 서울열린데이터광장
현재 URL: https://data.seoul.go.kr/dataList/OA-22300/F/1/datasetView.do


In [19]:
from selenium.webdriver.common.by import By

filename = "seoul_purpose_admdong3_20260831.zip"

elements = driver.find_elements(
    By.XPATH,
    f"//*[contains(text(), '{filename}')]"
)

print("찾은 개수:", len(elements))

for i, e in enumerate(elements[:5]):
    print(i, e.tag_name, e.text)

찾은 개수: 2
0 span seoul_purpose_admdong3_20260831.zip
1 span 


In [20]:
file_element = elements[0]

row = file_element.find_element(
    By.XPATH,
    "./ancestor::tr"
)

print("행 텍스트:")
print(row.text)

print("\n클릭 가능한 요소:")

clickables = row.find_elements(
    By.XPATH,
    ".//a | .//button | .//input"
)

for i, e in enumerate(clickables):
    print(
        i,
        "tag =", e.tag_name,
        "text =", e.text,
        "href =", e.get_attribute("href"),
        "onclick =", e.get_attribute("onclick")
    )

행 텍스트:
1 seoul_purpose_admdong3_20260831.zip

클릭 가능한 요소:
0 tag = a text =  href = javascript:downloadFile('260831'); onclick = None


In [21]:
import os
import time

DOWNLOAD_DIR = "/content/seoul_mobility"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# 다운로드 위치 지정
driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": DOWNLOAD_DIR
    }
)

# 방금 찾은 다운로드 링크 클릭
clickables[0].click()

print("다운로드 클릭 완료")

# 10초 정도 기다리기
time.sleep(10)

print("현재 파일:")
print(os.listdir(DOWNLOAD_DIR))

다운로드 클릭 완료
현재 파일:
['seoul_purpose_admdong3_20260831.zip']


In [22]:
import os
import zipfile

file_path = "/content/seoul_mobility/seoul_purpose_admdong3_20260831.zip"

# 파일 크기
size_mb = os.path.getsize(file_path) / (1024 * 1024)

print(f"파일 크기: {size_mb:.2f} MB")

# ZIP 정상 여부 확인
with zipfile.ZipFile(file_path, "r") as z:
    bad_file = z.testzip()

    if bad_file is None:
        print("ZIP 파일 정상")
    else:
        print("손상된 파일:", bad_file)

    print("ZIP 내부 파일:")
    print(z.namelist())

파일 크기: 68.71 MB
ZIP 파일 정상
ZIP 내부 파일:
['seoul_purpose_admdong3_final_20260831.csv']


In [23]:
import os
import time

test_dates = [
    "260830",
    "260829",
    "260828"
]

for date_code in test_dates:

    filename = f"seoul_purpose_admdong3_20{date_code}.zip"
    filepath = os.path.join(DOWNLOAD_DIR, filename)

    print(f"다운로드 시작: {filename}")

    # 사이트의 JavaScript 다운로드 함수 직접 실행
    driver.execute_script(
        f"downloadFile('{date_code}');"
    )

    # 다운로드 완료까지 대기
    while True:

        downloading = any(
            f.endswith(".crdownload")
            for f in os.listdir(DOWNLOAD_DIR)
        )

        if os.path.exists(filepath) and not downloading:
            break

        time.sleep(2)

    size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(f"완료: {filename} ({size_mb:.2f} MB)")

다운로드 시작: seoul_purpose_admdong3_20260830.zip
완료: seoul_purpose_admdong3_20260830.zip (58.85 MB)
다운로드 시작: seoul_purpose_admdong3_20260829.zip
완료: seoul_purpose_admdong3_20260829.zip (67.00 MB)
다운로드 시작: seoul_purpose_admdong3_20260828.zip
완료: seoul_purpose_admdong3_20260828.zip (75.30 MB)


In [24]:
import os
import zipfile

folder = "/content/seoul_mobility"

for f in sorted(os.listdir(folder)):

    if not f.endswith(".zip"):
        continue

    path = os.path.join(folder, f)

    try:
        with zipfile.ZipFile(path, "r") as z:
            bad = z.testzip()

            if bad is None:
                print(f"정상: {f}")
            else:
                print(f"손상: {f}")

    except Exception as e:
        print(f"오류: {f} -> {e}")

정상: seoul_purpose_admdong3_20260828.zip
정상: seoul_purpose_admdong3_20260829.zip
정상: seoul_purpose_admdong3_20260830.zip
정상: seoul_purpose_admdong3_20260831.zip


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
import os

DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

저장 폴더: /content/drive/MyDrive/seoul_mobility


In [ ]:
import os
import time

DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"

driver.execute_cdp_cmd(
    "Page.setDownloadBehavior",
    {
        "behavior": "allow",
        "downloadPath": DOWNLOAD_DIR
    }
)

date_code = "260827"
filename = "seoul_purpose_admdong3_20260827.zip"
filepath = os.path.join(DOWNLOAD_DIR, filename)

driver.execute_script(
    f"downloadFile('{date_code}');"
)

print("다운로드 시작:", filename)

while True:

    downloading = any(
        f.endswith(".crdownload")
        for f in os.listdir(DOWNLOAD_DIR)
    )

    if os.path.exists(filepath) and not downloading:
        break

    time.sleep(2)

size_mb = os.path.getsize(filepath) / (1024 * 1024)

print(f"다운로드 완료: {filename} ({size_mb:.2f} MB)")

### 1년치 다운로드

In [ ]:
from datetime import datetime, timedelta

START_DATE = datetime(2025, 9, 1)
END_DATE   = datetime(2026, 8, 31)

dates = []

current = START_DATE

while current <= END_DATE:
    dates.append(current)
    current += timedelta(days=1)

print("전체 날짜 수:", len(dates))
print("첫 날짜:", dates[0])
print("마지막 날짜:", dates[-1])

In [ ]:
import os
import time
import zipfile

DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"

success = []
skipped = []
failed = []

for i, date in enumerate(dates, start=1):

    # 사이트 downloadFile()에 들어가는 형식: YYMMDD
    date_code = date.strftime("%y%m%d")

    # 실제 파일명: YYYYMMDD
    date_full = date.strftime("%Y%m%d")

    filename = f"seoul_purpose_admdong3_{date_full}.zip"
    filepath = os.path.join(DOWNLOAD_DIR, filename)

    print(f"[{i:03d}/365] {filename}")

    # -----------------------------
    # 이미 존재하는 정상 ZIP은 건너뛰기
    # -----------------------------
    if os.path.exists(filepath):

        try:
            with zipfile.ZipFile(filepath, "r") as z:
                if z.testzip() is None:
                    print("   이미 존재 → 건너뜀")
                    skipped.append(filename)
                    continue
        except:
            # 깨진 파일이면 삭제 후 다시 다운로드
            os.remove(filepath)

    try:

        # 다운로드 실행
        driver.execute_script(
            f"downloadFile('{date_code}');"
        )

        start_time = time.time()

        # -----------------------------
        # 다운로드 완료 대기
        # -----------------------------
        while True:

            partial = [
                f for f in os.listdir(DOWNLOAD_DIR)
                if f.endswith(".crdownload")
            ]

            if os.path.exists(filepath) and not partial:
                break

            # 15분 이상 걸리면 실패 처리
            if time.time() - start_time > 900:
                raise TimeoutError("다운로드 시간 초과")

            time.sleep(2)

        # -----------------------------
        # ZIP 정상 여부 확인
        # -----------------------------
        with zipfile.ZipFile(filepath, "r") as z:

            if z.testzip() is not None:
                raise Exception("ZIP 파일 손상")

        size_mb = os.path.getsize(filepath) / (1024 * 1024)

        print(f"   완료 → {size_mb:.2f} MB")

        success.append(filename)

        # 서버 요청 간격
        time.sleep(1)

    except Exception as e:

        print("   실패 →", e)

        failed.append(filename)

### 데이터 완전성확인

In [30]:
import os

DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"

zip_files = [
    f for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".zip")
]

print("ZIP 파일 개수:", len(zip_files))

ZIP 파일 개수: 365


In [31]:
import os
import zipfile

DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"

zip_files = sorted([
    f for f in os.listdir(DOWNLOAD_DIR)
    if f.endswith(".zip")
])

normal = []
damaged = []

for i, filename in enumerate(zip_files, start=1):

    path = os.path.join(DOWNLOAD_DIR, filename)

    try:
        with zipfile.ZipFile(path, "r") as z:
            bad_file = z.testzip()

            if bad_file is None:
                normal.append(filename)
            else:
                damaged.append(filename)

    except Exception:
        damaged.append(filename)

    if i % 50 == 0:
        print(f"{i}개 검사 완료")

print()
print("정상 ZIP:", len(normal))
print("문제 ZIP:", len(damaged))

if damaged:
    print("\n문제 있는 파일:")
    for f in damaged:
        print(f)

50개 검사 완료
100개 검사 완료
150개 검사 완료
200개 검사 완료
250개 검사 완료
300개 검사 완료
350개 검사 완료

정상 ZIP: 365
문제 ZIP: 0


# 데이터 확인

In [32]:
# import os
# import zipfile

# DOWNLOAD_DIR = "/content/drive/MyDrive/seoul_mobility"

# sample_zip = os.path.join(
#     DOWNLOAD_DIR,
#     "seoul_purpose_admdong3_20260831.zip"
# )

# with zipfile.ZipFile(sample_zip, "r") as z:

#     print("ZIP 내부 파일:")
#     for info in z.infolist():
#         print()
#         print("파일명:", info.filename)
#         print("압축 크기:", round(info.compress_size / 1024 / 1024, 2), "MB")
#         print("압축 해제 크기:", round(info.file_size / 1024 / 1024, 2), "MB")
#         print(
#             "압축률:",
#             round(info.file_size / info.compress_size, 2),
#             "배"
#         )

ZIP 내부 파일:

파일명: seoul_purpose_admdong3_final_20260831.csv
압축 크기: 68.71 MB
압축 해제 크기: 406.98 MB
압축률: 5.92 배


In [35]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        sample_df = pd.read_csv(
            f,
            nrows=5
        )

print("컬럼 수:", len(sample_df.columns))

print("\n컬럼명:")
for col in sample_df.columns:
    print(col)

print("\n샘플 데이터:")
display(sample_df)

컬럼 수: 11

컬럼명:
o_admdong_cd
d_admdong_cd
st_time_cd
fns_time_cd
in_forn_div_nm
forn_citiz_nm
move_purpose
move_dist
move_time
cnt
etl_ymd

샘플 데이터:


,o_admdong_cd,d_admdong_cd,st_time_cd,fns_time_cd,in_forn_div_nm,forn_citiz_nm,move_purpose,move_dist,move_time,cnt,etl_ymd
0,11200645,11140615,820,840,내국인,한국,1,1570,14,2.92,20260831
1,11200645,11140615,820,840,내국인,한국,7,1850,27,5.83,20260831
2,11200645,11140615,840,900,내국인,한국,1,2151,28,4.14,20260831
3,11200645,11140615,900,900,내국인,한국,1,2080,6,3.16,20260831
4,11200645,11140615,900,900,내국인,한국,7,2099,17,2.92,20260831


In [34]:
print("자료형")
print(sample_df.dtypes)

print("\n결측치")
print(sample_df.isnull().sum())

자료형
o_admdong_cd        int64
d_admdong_cd        int64
st_time_cd          int64
fns_time_cd         int64
in_forn_div_nm     object
forn_citiz_nm      object
move_purpose        int64
move_dist           int64
move_time           int64
cnt               float64
etl_ymd             int64
dtype: object

결측치
o_admdong_cd      0
d_admdong_cd      0
st_time_cd        0
fns_time_cd       0
in_forn_div_nm    0
forn_citiz_nm     0
move_purpose      0
move_dist         0
move_time         0
cnt               0
etl_ymd           0
dtype: int64


In [36]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

total_rows = 0
missing = None

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            chunksize=200_000,
            dtype={
                "o_admdong_cd": "string",
                "d_admdong_cd": "string"
            }
        ):

            total_rows += len(chunk)

            chunk_missing = chunk.isnull().sum()

            if missing is None:
                missing = chunk_missing
            else:
                missing += chunk_missing


print("전체 행 수:", f"{total_rows:,}")

print("\n전체 결측치:")
print(missing)

전체 행 수: 6,134,711

전체 결측치:
o_admdong_cd      0
d_admdong_cd      0
st_time_cd        0
fns_time_cd       0
in_forn_div_nm    0
forn_citiz_nm     0
move_purpose      0
move_dist         0
move_time         0
cnt               0
etl_ymd           0
dtype: int64


In [37]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

time_values = set()
purpose_values = set()
origin_codes = set()
dest_codes = set()

cnt_min = float("inf")
cnt_max = float("-inf")
cnt_sum = 0

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            chunksize=200_000,
            dtype={
                "o_admdong_cd": "string",
                "d_admdong_cd": "string"
            }
        ):

            time_values.update(chunk["st_time_cd"].unique())
            purpose_values.update(chunk["move_purpose"].unique())

            origin_codes.update(chunk["o_admdong_cd"].unique())
            dest_codes.update(chunk["d_admdong_cd"].unique())

            cnt_min = min(cnt_min, chunk["cnt"].min())
            cnt_max = max(cnt_max, chunk["cnt"].max())
            cnt_sum += chunk["cnt"].sum()


print("출발 시간 코드:")
print(sorted(time_values))

print("\n이동목적 코드:")
print(sorted(purpose_values))

print("\n출발 행정동 코드 수:", len(origin_codes))
print("도착 행정동 코드 수:", len(dest_codes))

print("\ncnt 최소값:", cnt_min)
print("cnt 최대값:", cnt_max)
print("하루 총 이동인구:", round(cnt_sum, 2))

# 1	출근
# 2	등교
# 3	귀가
# 4	쇼핑
# 5	관광
# 6	병원
# 7	기타

출발 시간 코드:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(700), np.int64(720), np.int64(740), np.int64(800), np.int64(820), np.int64(840), np.int64(900), np.int64(920), np.int64(940), np.int64(1700), np.int64(1720), np.int64(1740), np.int64(1800), np.int64(1820), np.int64(1840), np.int64(1900), np.int64(1920), np.int64(1940)]

이동목적 코드:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

출발 행정동 코드 수: 658
도착 행정동 코드 수: 658

cnt 최소값: 1.0
cnt 최대값: 22237.8
하루 총 이동인구: 61457247.65


### 시간코드별 데이터 건수 확인

In [38]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

time_counts = {}

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            usecols=["st_time_cd"],
            chunksize=200_000
        ):

            counts = chunk["st_time_cd"].value_counts()

            for time_code, count in counts.items():
                time_counts[time_code] = (
                    time_counts.get(time_code, 0) + count
                )

time_counts = pd.Series(time_counts).sort_index()

print(time_counts)

# 0~6, 10~16, 20~23 → 1시간 단위
# 700, 720, 740 → 07:00, 07:20, 07:40
# 800, 820, 840 → 08:00, 08:20, 08:40
# 900, 920, 940 → 09:00, 09:20, 09:40
# 1700~1940도 같은 방식으로 17~19시를 20분 단위

0        25337
1        44084
2        40050
3        45193
4        80163
5       188656
6       360884
10      252030
11      253056
12      270428
13      277313
14      282121
15      304138
16      370881
20      190705
21      151301
22       83577
23       12558
700     196787
720     206928
740     205654
800     199508
820     183583
840     159342
900     146550
920     128920
940     113656
1700    184518
1720    177491
1740    172815
1800    212776
1820    153690
1840    132472
1900    127708
1920    105361
1940     94477
dtype: int64


In [39]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

prefix_counts = {}

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            usecols=["d_admdong_cd"],
            dtype={"d_admdong_cd": "string"},
            chunksize=200_000
        ):

            # 앞 두 자리 추출
            prefix = chunk["d_admdong_cd"].str[:2]

            counts = prefix.value_counts()

            for p, cnt in counts.items():
                prefix_counts[p] = prefix_counts.get(p, 0) + cnt


result = pd.Series(prefix_counts).sort_index()

print(result)

# 전체 행 수        6,134,711
# 서울 도착 행 수   4,040,557

11    4040557
12      10786
26       7079
27       6370
28     306936
30      14260
31       2262
36       6190
41    1575345
43      36552
44      55109
47      13581
48       6595
50       2043
51      39772
52      11274
dtype: int64


In [40]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

seoul_dongs = set()
seoul_cnt_sum = 0
seoul_rows = 0

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            usecols=["d_admdong_cd", "cnt"],
            dtype={"d_admdong_cd": "string"},
            chunksize=200_000
        ):

            # 서울 도착지만 선택
            seoul = chunk[
                chunk["d_admdong_cd"].str.startswith("11")
            ]

            seoul_rows += len(seoul)

            seoul_dongs.update(
                seoul["d_admdong_cd"].unique()
            )

            seoul_cnt_sum += seoul["cnt"].sum()


print("서울 도착 행 수:", f"{seoul_rows:,}")
print("서울 도착 행정동 수:", len(seoul_dongs))
print("서울 도착 총 이동인구:", f"{seoul_cnt_sum:,.2f}")

# 서울 도착 행 수: 4,040,557
# 서울 도착 행정동 수: 427
# 서울 도착 총 이동인구: 22,161,911.81

서울 도착 행 수: 4,040,557
서울 도착 행정동 수: 427
서울 도착 총 이동인구: 22,161,911.81


In [41]:
import pandas as pd
import zipfile
import os

sample_zip = os.path.join(
    DOWNLOAD_DIR,
    "seoul_purpose_admdong3_20260831.zip"
)

dong_totals = {}

with zipfile.ZipFile(sample_zip, "r") as z:

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:

        for chunk in pd.read_csv(
            f,
            usecols=["d_admdong_cd", "cnt"],
            dtype={"d_admdong_cd": "string"},
            chunksize=200_000
        ):

            # 서울 도착지만
            seoul = chunk[
                chunk["d_admdong_cd"].str.startswith("11")
            ]

            # 행정동별 유입 이동량 합계
            grouped = (
                seoul
                .groupby("d_admdong_cd")["cnt"]
                .sum()
            )

            for dong, value in grouped.items():
                dong_totals[dong] = (
                    dong_totals.get(dong, 0)
                    + value
                )


daily_rank = (
    pd.Series(dong_totals, name="유입이동인구")
    .sort_values(ascending=False)
    .reset_index()
)

daily_rank.columns = [
    "행정동코드",
    "유입이동인구"
]

print(daily_rank.head(20))

       행정동코드     유입이동인구
0   11560540  362872.09
1   11680640  339929.74
2   11110615  256847.96
3   11545510  223901.70
4   11140550  221275.60
5   11440660  199884.22
6   11650530  193306.08
7   11170625  189078.64
8   11680580  172053.69
9   11110640  162900.81
10  11410585  161479.37
11  11710642  153070.84
12  11500603  147904.73
13  11650581  143361.59
14  11710710  137172.80
15  11680650  133726.46
16  11410565  132752.11
17  11560535  132032.68
18  11740685  128742.61
19  11680531  128405.43
